In [ ]:
%matplotlib inline
import os
import sys
from os.path import join as pjoin
from tifffile import imread, imwrite, TiffFile
import numpy as np
import shutil
import matplotlib.pyplot as plt
from glob import glob
import pandas as pd
import cv2
from tqdm import tqdm
import subprocess
from scipy.ndimage import gaussian_filter,median_filter
from scipy.interpolate import interp1d
from scipy.signal import detrend, butter, filtfilt

project_root = '/home/lsh/WF_GoNogo'
if project_root not in sys.path:
    sys.path.append(project_root)
from utils.wfield_utils import *

### detect pupil size, positions, facial movements

In [ ]:
config_path = "/home/lsh/WF_GoNogo/config/A095_config.yaml"

In [ ]:
preprocess_mice(config_path)

In [ ]:
config = load_config(config_path)
for session in config["sessions"]:
    print(f"Processing session: {session}")
    rawPath = config["paths"]["raw"].format(
        base_dir=config["base_dir"], mouse_id=config["mouse_id"], session=session
    )
    print(f"Raw path: {rawPath}")
    processPath = config["paths"]["process"].format(
        base_dir=config["base_dir"], mouse_id=config["mouse_id"], session=session
    )
    os.makedirs(processPath, exist_ok=True)
    qc_path = config["paths"]["qc"].format(
        base_dir=config["base_dir"], mouse_id=config["mouse_id"], session=session
    )
    os.makedirs(qc_path, exist_ok=True)
    timePath = config["paths"]["time"].format(
        base_dir=config["base_dir"], mouse_id=config["mouse_id"], session=session
    )
    os.makedirs(timePath, exist_ok=True)

    items = glob(pjoin(rawPath, '202?????-??????-4*'))
    experiments = list(set([os.path.basename(item)[:15] for item in items]))
    print("Experiments found:", experiments)

    for experiment in experiments:
        # 1. Merge two channels
        merged_file = merge_two_channels(rawPath, processPath, timePath, experiment)

        # 2. Outlier detection and correction
        values_470 = np.loadtxt(pjoin(processPath, f"{experiment}-470-Values.csv"), delimiter=',')
        values_405 = np.loadtxt(pjoin(processPath, f"{experiment}-405-Values.csv"), delimiter=',')
        n_frames = min(len(values_470), len(values_405))
        mean_values = np.stack([values_470[:n_frames], values_405[:n_frames]], axis=1)
        out470, out405 = detect_outlier(mean_values, std_thr=config["preprocess"]["outlier_std"], qc_path=qc_path, plot=True)
        # out470, out405 = detect_outlier(mean_values, std_thr=2, plot=True)
        corrected_file = correct_lum_outlier(merged_file, out470, out405, plot=True, qc_path=qc_path)
        print("Outlier removal done")

        # 3. mask
        mask_path = os.path.join(qc_path, 'brain_mask.npy')
        masked_file = apply_mask_and_save_bin(
            corrected_file=corrected_file,
            mask_path=mask_path,
            qc_path=qc_path,
            chunk_size=256
        )
        
        
        # 4. SVD
        U_list, V_list, im_avg_list = svd_widefield(processPath, experiment, masked_file , downsample_factor=15, max_components=500)
print(f"all done")